# 23 · 和别人比：让现成的诊断工具考同一份题

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → [23 和别人比 ✅]
```

> **这一章的代码在合并请求 #25 里。** 这一章不花钱：对手 LIRICAL 是在本机运行的程序，不用大模型。留出集上一共跑了 4 次，都记在 MLflow 里。运行这本笔记只读记录；第 2、3 步会真的运行几次 LIRICAL（每次约 5 秒，需要 Java，没有就自动跳过）。
>
> 和第 21 章一样，**这一章分两段写**：第 1–5 步是考之前写的，第 6 步起是考完以后填的结果。

## 背景：自己出题、自己当对手，不算数

第 21 章的结论是"agent 比固定流程强"。可是固定流程、传统排名、先排名再核对，这些对照系统**全是我们自己写的**。

这里有个很现实的问题：对照是自己写的，写得弱一点，自己的系统就显得强。不一定是故意的，但别人没法分辨。行业里管弱的对照叫"稻草人"（strawman baseline）。

所以别人一定会问第二个问题：

> **和已经发表、大家在用的工具比，怎么样？**

这一章回答它。对手是 **LIRICAL**：杰克逊实验室（就是做 HPO 的团队）2020 年发表的罕见病诊断工具，一直在维护。

但这里有一个坑，也是这一章最重要的内容：**不能直接拿它论文里的分数来比。** 它的评测是在"不遮挡"的条件下做的，也就是病人自己那篇论文还在知识库里。我们的分数是遮挡以后的。两个数不是在同一种条件下量的，不能放在一起。

唯一公平的做法：**把它搬进我们的考场，同样的题，同样的知识库，同样的遮挡，重新考一次。**

**公司里的做法**：上线一个新系统之前，必须和"现在线上那个"或者"业界公认的那个"在同一份数据、同一套规则下比，叫**对标**（benchmark against the incumbent）。只和自己的旧版本比，评审是不认的。

## 这一章做的 8 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 选谁来比 | 三个筛选条件 | 对手要够强，并且能在同样条件下考 |
| 2 | 把别人的工具接进我们的考场 | 适配器；版本和校验码登记 | 它和我们的系统走同一套考试、判分、记录 |
| 3 | 给它同样的遮挡 | 每道题单独写一份遮过的知识库文件 | 它也读不到考题自己的论文 |
| 4 | 先写方案 | 预注册：`docs/comparison_plan.md` | 不看了结果再挑怎么比 |
| 5 | 在开发集上调通 | dev-200 | 留出集只用来出最后的数 |
| 6 | 留出集：agent 对 LIRICAL | 置信区间、逐题配对检验 | 回答方案里的主要问题 |
| 7 | 不遮挡会怎样 | 同一个工具，遮和不遮各考一次 | 量出"答案就在知识库里"值多少分 |
| 8 | 和考前的预期对照；下一步 | | 知道自己哪里想错了 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 选谁来比

### 在干什么
从现成的罕见病诊断工具里，挑出这一轮能公平比较的。

### 背景
能叫得出名字的工具不少：Exomiser、LIRICAL、Phenomizer、PubCaseFinder、PhenoBrain、Phen2Gene，再加上通用大模型和 DeepRare。不可能、也没必要全比。挑错了对手，结论没人信：挑弱的是稻草人，挑一个没法在同样条件下考的，比出来的数字没有意义。

### 用了什么技术，为什么
三个筛选条件：

| 条件 | 为什么 |
|---|---|
| **够强、公认** | 发表过，有人在用，还在维护 |
| **做的是同一件事** | 只看症状，给出疾病的排名 |
| **能搬进我们的考场** | 能在本机运行，并且它读的知识库我们能替它遮挡 |

第三条筛掉的最多：

| 工具 | 结论 | 原因 |
|---|---|---|
| **LIRICAL** | **比** | 三条都满足。本机运行，读的就是 HPO 的那个标注文件 |
| Exomiser | 这一轮不比 | 要读病人的测序文件（VCF）。我们的病例只有致病基因的名字，没有测序文件 |
| PubCaseFinder、Phenomizer、PhenoBrain | 不比 | 是网站。它们的资料库里有这些已发表的病例，我们没法替它们遮掉 |
| Phen2Gene | 不比 | 排的是基因，不是病；知识库也没法遮 |
| 通用大模型、DeepRare | 下一轮 | 要花钱，单独定方案和预算 |

**LIRICAL 的方法和我们的不一样**，这让比较更有意思：

- 我们的排名工具（第 10 章）算**语义相似度**：病人的症状集合和疾病的症状集合有多像
- LIRICAL 算**似然比**：对每一个症状，"得这个病的人有它"比"一般的病人有它"多见多少倍，全部乘起来，换算成一个概率。医生看化验单时用的就是这种推理

### 公司里什么情况怎么做
- **对标的对象要写明版本**：LIRICAL 2.4.1，不是"LIRICAL"。工具会更新，分数会变
- **说清楚没比谁、为什么**：比不了的原因写出来，比假装它们不存在要可信

### 面试怎么说
> "My own baselines can't answer whether the agent is good — I wrote them. So I compared against LIRICAL, a published and maintained phenotype-driven tool with a different method, likelihood ratios rather than semantic similarity. I picked it because it runs offline on the same HPO annotation file, which means I can put it under the same leakage controls; web tools like PubCaseFinder I left out because their databases can't be masked."

---

## 第 2 步 · 把别人的工具接进我们的考场

### 在干什么
让 LIRICAL 变成考试程序眼里的"又一个系统"：收到一道题，交回一个排名。

### 背景
LIRICAL 是一个 Java 写的命令行程序，和我们的 Python 代码完全是两个世界。如果在考场外面单独跑它、自己另写一个脚本算分，那么"题目是不是同一份、判分规则是不是同一套、有没有做泄露审计"都得另外保证，哪一处不一样，比较就不成立。

### 用了什么技术，为什么

**适配器**（wrapper / adapter）：一小段代码，站在考试程序和 LIRICAL 中间。对考试程序，它和别的系统长得一样（`predict(题目) → 排名`）；对 LIRICAL，它负责准备输入、启动程序、读回结果。打个比方：给外国来的考生配一个翻译，卷子和评分标准一个字不改。

代码在 `src/raredx/baselines/lirical.py`。每道题它做四件事：

1. 整理症状（作废的编号换成现行的，去掉不是症状的条目），和我们自己的工具用同一套整理方法
2. 准备 LIRICAL 要读的数据文件夹（下一步细讲）
3. 启动 LIRICAL：`java -jar lirical-cli.jar prioritize --observed-phenotypes HP:...,HP:... --sex FEMALE ...`
4. 读回它输出的表格，取 OMIM 疾病的前 10 名

**版本和来源登记**：LIRICAL 的安装包和别的数据一样，登记在 `evals/datasets.toml` 里：版本号、下载地址、校验码。下载下来校验码对不上就拒绝使用。LIRICAL 本身不放进 git（它的许可只允许非商业使用，我们只是拿来做研究对比）。

**它的排名被当成"工具输出"记进追踪**：这样第 17 章做的泄露审计会自动去读它，检查里面有没有出现不该出现的病。

### 公司里什么情况怎么做
- **对手也要进同一条评测流水线**：不允许"我们的系统用评测平台跑，对手的数字是另一个人用另一个脚本算的"
- **第三方工具当成依赖管理**：固定版本、核对校验码、记录许可

### 执行
登记的内容，和安装好以后记录下来的文件清单：

In [ ]:
run("grep -A 9 '^\\[lirical\\]' evals/datasets.toml")
build = ROOT / "data/processed/lirical/2.4.1/build.json"
if build.exists():
    info = __import__("json").loads(build.read_text())
    print("装好的 LIRICAL：版本", info["lirical"], "   删掉的重复基因行：", info["gene_rows_dropped"])
    print("它用的文件（每个都记了校验码）：", ", ".join(info["files"]))
else:
    print("这台电脑上还没有安装 LIRICAL。要装的话（约 400 MB，需要 Java 17）：")
    print("   uv run raredx data fetch lirical hpo-json && uv run raredx data lirical")

真的跑一次：晶状体脱位、蜘蛛指、主动脉根部瘤（马凡综合征的经典三联征，第 22 章用过）。

In [ ]:
from raredx import cli
from raredx.baselines import lirical
from raredx.config import settings

LIRICAL = None
try:
    hpo = cli._knowledge(ROOT / "evals", ROOT / "data")
    LIRICAL = lirical.Lirical(hpo, ROOT / "data/processed/lirical/2.4.1", lirical.find_java(settings.java))
except (lirical.LiricalNotSetUp, lirical.JavaNotFound) as e:
    print("跳过（LIRICAL 没有装好）：", e)
if LIRICAL is not None:
    import time
    t = time.time()
    rows = LIRICAL.ranked(["HP:0001083", "HP:0001166", "HP:0002616"], "MALE")
    print(f"LIRICAL 用了 {time.time() - t:.1f} 秒，给 {len(rows)} 个病排了名。前 5 个：")
    for i, r in enumerate(rows[:5], 1):
        print(f"   {i}. {r['disease_name']}（{r['disease_id']}）   概率 {r['posttest_probability']}")

### 结果怎么看
- 登记里有版本 `2.4.1`、下载地址、许可（只限非商业使用）和安装包的校验码
- 安装记录里写着"删掉的重复基因行：3"，见下面的坑
- 真跑一次约 5 秒。前 3 名是 `Marfanoid habitus with situs inversus`、`Marfan lipodystrophy syndrome`、`Marfan syndrome`，概率都在 99% 以上。它算出的"概率"很自信，但那是在"病人一定得了这八千多种病里的一种、每个症状互相独立"的假设下算的，不能当真实的把握来读

> **一个真实踩过的坑：别人的工具，一启动就崩溃**
>
> LIRICAL 装好以后第一次运行，直接报错退出：`Duplicate key NCBIGene:ENSG00000230417`。原因：它启动时要读一份人类基因名录（HGNC），这份名录是它自己去下载的"最新版"。最新版里恰好有 3 对基因共用一个编号，LIRICAL 2.4.1 处理不了。
>
> 也就是说：**工具本身没变，它依赖的上游数据变了，它就坏了。** 这正是我们从第 05 章起坚持"所有数据固定版本、核对校验码"的原因。
>
> 处理办法：把那 3 行重复的记录删掉。然后核对了一件事：基因名录会不会影响只看症状的排名？把名录砍到只剩 200 行再跑，排名和原来**一字不差**。所以这个修补不影响比较，并且写进了方案和每次考试的记录里。

### 面试怎么说
> "LIRICAL runs behind an adapter that makes it one more system under the same harness — same manifest, same scorer, same leak audit over its output — instead of a side script with its own scoring. Its release is pinned by checksum in the dataset registry. It crashed on first run because an upstream gene table it downloads had changed; I repaired three rows, verified the phenotype ranking was byte-identical with and without the gene data, and recorded the repair in the run's parameters."

---

## 第 3 步 · 给它同样的遮挡

### 在干什么
让 LIRICAL 和我们的系统一样，读不到考题自己那篇论文里的内容。

### 背景
回顾一下泄露是怎么回事（第 10 章）：我们的考题是已发表论文里的病人；而 HPO 记录"每种病有哪些症状"时，资料来源恰恰也包括这些论文。所以一道题的答案，很可能就写在知识库里，而且是从这个病人自己身上抄下来的。

我们自己的工具从第 10 章起就做了遮挡。LIRICAL 是别人的程序，它不知道"遮挡"这回事，会把知识库文件整个读进去。

### 用了什么技术，为什么

办法很直接：**不改 LIRICAL，改它读的文件。**

LIRICAL 从一个叫 `phenotype.hpoa` 的文件里读"每种病有哪些症状"。这个文件有 28 万多行，每一行是一条记录："某个病 — 有某个症状 — 出处是哪篇文献"。我们对每一道题，现写一份遮过的文件给它，用的是同样的三条规则（`docs/leakage.md`）：

| 规则 | 在文件里怎么体现 |
|---|---|
| 只靠这道题自己的论文的记录，遮掉 | 这些行删掉 |
| 别的文献也支持的记录，保留 | 这些行留着 |
| 只在这篇论文里描述过的病，当它不存在 | 这个病的所有行都删掉 |

写一份文件约 0.5 秒。840 道题就是写 840 份，每份只在这道题运行时存在，用完就删。

**为什么不干脆做一份"把所有考题的论文都遮掉"的文件，一次搞定**：那样每道题会被多遮掉 839 篇别人的论文，知识库被削得太多，对 LIRICAL 不公平。我们的系统是"每道题只遮自己的那一篇"，它也必须一样。

### 公司里什么情况怎么做
- **对手和自己必须在同一种信息条件下比**。最常见的作弊（往往是无意的）就是：自己的系统做了严格的数据隔离，对手的数字直接抄论文
- **改输入，不改对手的程序**：一改它的代码，它就不是那个"公认的工具"了

### 执行
拿两个真实的病人看，遮挡到底从文件里删掉了什么：

In [ ]:
import tempfile
from raredx.service import examples as example_store
from raredx.tools.check_match import CheckMatch

hpo = cli._knowledge(ROOT / "evals", ROOT / "data")
_, cases = cli._processed(ROOT / "data", ROOT / "evals")
by_id = {c.id: c for c in cases}
listed = [by_id[case_id] for case_id, _, _ in example_store.listed()]
patient, unanswerable = listed[1], listed[4]          # 第 22 章的示例 2 和示例 5

source = ROOT / "data/raw/hpo/v2026-09-01/phenotype.hpoa"
everything = [l for l in source.read_text().splitlines() if not l.startswith("#")][1:]
print(f"知识库文件一共 {len(everything):,} 行记录\n")
for case in (patient, unanswerable):
    with tempfile.TemporaryDirectory() as tmp:
        hpo.write_annotations(Path(tmp) / "masked.hpoa", frozenset({case.pmid}))
        kept = set((Path(tmp) / "masked.hpoa").read_text().splitlines())
    removed = [l.split("\t") for l in everything if l not in kept]
    diseases = sorted({(r[0], r[1]) for r in removed})
    left = len(hpo.lines(case.disease_id, "P", frozenset({case.pmid})))
    print(f"病人 {case.pmid}（论文确诊：{case.disease_label}）")
    print(f"   遮掉她/他自己的论文：删掉 {len(removed)} 行，全部属于：{diseases}")
    print(f"   这个病在文件里还剩 {left} 行症状记录" + ("  ← 一行不剩：对任何工具来说，这个病不存在了" if left == 0 else ""))
    check = CheckMatch(hpo)
    from raredx.knowledge import masking
    seen = len(check(case.hpo_ids, case.disease_id).explained)
    with masking.masked([case.pmid]):
        try:
            hidden = f"{len(check(case.hpo_ids, case.disease_id).explained)}"
        except Exception:
            hidden = "查不到这个病"
    print(f"   病人的 {len(case.hpo_ids)} 个表现，这个病对得上几个：不遮 {seen} 个 → 遮了以后 {hidden}\n")

同一个病人，让 LIRICAL 真的各考一次：遮和不遮，确诊的病分别排第几？

In [ ]:
if LIRICAL is None:
    print("跳过（LIRICAL 没有装好）。")
else:
    for case in (patient, unanswerable):
        open_book = [r["disease_id"] for r in LIRICAL.ranked(case.hpo_ids, case.sex)]
        with masking.masked([case.pmid]):
            closed_book = [r["disease_id"] for r in LIRICAL.ranked(case.hpo_ids, case.sex)]
        place = lambda ranking: f"第 {ranking.index(case.disease_id) + 1} 名" if case.disease_id in ranking else "前 50 名里没有"
        print(f"{case.pmid}（{case.disease_label}）：不遮 → {place(open_book)}      遮 → {place(closed_book)}")

### 结果怎么看

**第一个病人**（PMID:25681079，就是第 22 章那位半乳糖血症病人）：

- 遮掉她自己的论文，文件里删掉 9 行，**这 9 行全都属于"半乳糖血症"这个病**。也就是说，HPO 里半乳糖血症的症状记录，有 9 条是从她这篇论文里整理出来的
- 她有 8 个表现。不遮：半乳糖血症 8 个全对得上。遮了：只对得上 1 个
- LIRICAL：不遮，半乳糖血症排**第 1 名**；遮了，**前 50 名里没有**

8 个全对得上，不是因为半乳糖血症的典型表现恰好是这 8 个，而是因为**知识库里的那几条记录，就是照着她写的**。不遮挡的时候，工具做的其实是"在知识库里找到这个病人自己"。

**第二个病人**（PMID:20375004，僵硬皮肤综合征）：

- 删掉 15 行，是这个病在 HPO 里的全部记录。遮了以后，这个病一行不剩
- 不遮：7 个表现全对得上，LIRICAL 排第 1 名。遮了：这个病不存在，谁也答不出

这两位就是泄露的两种样子：一种是"答案的一部分是照着考题写的"，一种是"答案整个是照着考题写的"。

另外三位示例病人里，有两位的论文没有被 HPO 引用过，遮不遮结果完全一样（Holt-Oram 综合征两种情况下 LIRICAL 都排第 3）。泄露不是每道题都有，所以要到第 7 步在全部 840 道题上量。

### 面试怎么说
> "The off-the-shelf tool can't be told about masking, so I mask its input instead: for each case it reads its own copy of the HPO annotation file, written under the same three rules my tools apply — 0.5 seconds per case. For one held-out patient that removes nine lines, all nine for her own diagnosis: unmasked, the disease explains 8 of her 8 findings and LIRICAL ranks it first; masked, 1 of 8 and it is not in the top fifty."

---

## 第 4 步 · 先写方案

### 在干什么
和第 21 章一样，在 LIRICAL 碰留出集之前，把"怎么比、怎么算赢"写下来提交。

### 背景
这一次更需要预注册。因为结果有两种让人想"调整一下"的可能：LIRICAL 赢了，会想给自己找理由；LIRICAL 输得多，会想顺手把优势说得更大。先写好，这两条路都堵上。

### 用了什么技术，为什么
方案在 [`docs/comparison_plan.md`](../comparison_plan.md)，要点：

**主要结论只有一个**：同样遮挡的条件下，最终 agent 的第 1 名命中率是否显著高于 LIRICAL。LIRICAL 更高，或者不显著，都如实报告。

**公平的条件**逐条写明：同一份 840 道题；只给"病人有的表现"和性别；同一版 HPO（核对过：LIRICAL 自己下载的标注文件和我们固定的那一份，校验码完全相同）；同样的遮挡；只算 OMIM 疾病；默认设置，一个参数不调。

**跑 4 次，全部免费**：LIRICAL 遮；LIRICAL 不遮；我们两个不用模型的系统不遮（为了在"不遮"的条件下也能同等比较）。

**对 LIRICAL 做过的两处改动**也写在里面：删掉 3 行重复的基因记录；给 Java 加了两个省 CPU 的启动参数。两处都核对过，输出完全不变。

**考前的预期**：LIRICAL 遮挡后第 1 名 30% 上下；agent 领先约 10 个百分点；LIRICAL 不遮会到 65% 上下。

### 公司里什么情况怎么做
- **对手的设置谁来定**：用它的默认设置，或者请它的作者来调。自己替对手"调参"，调好调坏都说不清
- **在留出集上不调任何东西**，包括对手

### 执行
从 git 记录里看先后：方案和接入代码的提交时间，都早于留出集上的第一次运行。

In [ ]:
run("git log --format='%h %ad %s' --date=format:'%m-%d %H:%M' -- docs/comparison_plan.md src/raredx/baselines/lirical.py | tail -3")
run("uv run raredx eval runs --split heldout-papers 2>/dev/null | grep -E 'started|lirical|unmasked'")

### 结果怎么看
- 接入代码和方案是 10-13 晚上 22:47 提交的
- 留出集上 4 次运行的开始时间都在这之后，记录的代码版本是 `d3b8531`，就是接入代码的那次提交

### 面试怎么说
> "I pre-registered the comparison too: one primary question, the conditions that make it fair, the external tool at default settings with nothing tuned on held-out data, and my predictions — committed before LIRICAL saw a held-out case."

---

## 第 5 步 · 先在开发集上调通

### 在干什么
接入的代码，先在 20 道题（ci-20）和 200 道题（dev-200）上跑通、看一眼数字合不合理，再去碰留出集。

### 背景
接一个外部程序，一开始几乎一定有毛病：崩溃（第 2 步那个坑）、输出读错了列、太慢。这些都要调。如果拿留出集来调，留出集就被"看过"了。所以调试全部在开发集上做。

### 用了什么技术，为什么
没有新技术，就是第 06 章定下的规矩：**ci-20 看能不能跑通，dev-200 看数字，留出集只出最后的数。**

顺带解决了速度问题。LIRICAL 单独跑一道题 4 秒，但它一启动会开好几个线程做优化，很占 CPU；几道题同时跑就互相抢。加了两个启动参数（少做优化、垃圾回收只用一个线程）以后，CPU 用量减半，输出完全相同。

### 执行
dev-200 上 LIRICAL 的两次运行：

In [ ]:
run("uv run raredx eval runs --split dev-200 2>/dev/null | grep -E 'started|lirical' | cut -c1-110")

### 结果怎么看

| dev-200（200 道） | 第 1 名 | 前 10 名 |
|---|---|---|
| LIRICAL，遮 | 30.0% | 50.0% |
| LIRICAL，**不遮** | **67.0%** | 83.5% |
| 我们的"先排名再逐条核对"，遮 | 30.5% | 54.5% |
| 我们的最终 agent，遮 | 40.5% | 58.5% |

两件事在开发集上就已经很明显了：

1. **同一个工具，同样的题，遮不遮差了 37 个百分点**（67.0% 对 30.0%）。第 3 步那两位病人不是特例
2. 遮挡以后，LIRICAL 和我们自己写的"先排名再逐条核对"几乎一样（30.0% 对 30.5%）。这说明我们的对照不是稻草人：一个公认的工具，在同样条件下也就是这个水平

这些是开发集上的数，只用来确认接入没有问题、并写进方案的"预期"。能对外说的数在下一步。

### 面试怎么说
> "The integration was debugged on the dev set only. That is also where the headline effect first showed up: the same tool on the same 200 cases scores 67% top-1 with each case's own paper left in the knowledge, and 30% with it masked."

---

## 第 6 步 · 留出集：agent 对 LIRICAL

### 在干什么
回答方案里的主要问题：同样遮挡，最终 agent 的第 1 名是不是比 LIRICAL 强。

### 背景
LIRICAL 在留出集上考了一次（遮挡），840 道题全部作答，泄露审计通过。agent 用的是第 21 章那一次的记录，没有重考。

### 用了什么技术，为什么
和第 21 章一样：命中率加 95% 置信区间；同一批题逐题配对，双侧符号检验。

多了一样东西：**每道题的结果写成一个文件，放进 git**（`docs/comparison_hits.tsv`）。840 行，每行一道题，每个系统第 1 名、前 10 名答对没有（1 或 0）。

为什么要这个文件：MLflow 的记录库在这台电脑上，不在 git 里。别人拿到代码，看不到我们的考试记录，就没法核对数字。有了这个文件，任何人都能把这一章的每个数重新算一遍。下面的计算就是读它；最后再直接从 MLflow 的记录里算一次主要结论，两边对一下。

### 公司里什么情况怎么做
- **评测结果和代码一样要能复查**：汇总的数字之外，逐条的结果也要留档，这样别人能自己切分、自己验算
- **对标的结论要带配对检验**：两个系统的分数差 12 个百分点，还要说清楚是不是运气

### 执行

In [ ]:
import csv
from raredx.evals.scoring import p_value, sign_test, wilson

ROWS = list(csv.DictReader(open(ROOT / "docs/comparison_hits.tsv", encoding="utf-8"), delimiter="\t"))
NAMES = {"llm-direct": "大模型直接猜", "phenotype-rank": "传统方法", "lirical": "LIRICAL",
         "rank-then-check": "先排名再逐条核对", "fixed-pipeline": "固定流程", "agent": "最终 agent",
         "phenotype-rank (unmasked)": "传统方法（不遮）", "rank-then-check (unmasked)": "先排名再逐条核对（不遮）",
         "lirical (unmasked)": "LIRICAL（不遮）"}

def rate(system, k, rows=ROWS):
    n = sum(r[f"{system}@{k}"] == "1" for r in rows)
    lo, hi = wilson(n, len(rows))
    return f"{n / len(rows):6.1%} [{lo:.1%}–{hi:.1%}]"

def paired(a, b, k, rows=ROWS):
    wins = sum(r[f"{a}@{k}"] == "1" and r[f"{b}@{k}"] == "0" for r in rows)
    losses = sum(r[f"{b}@{k}"] == "1" and r[f"{a}@{k}"] == "0" for r in rows)
    return f"只有「{NAMES[a]}」对 {wins} 道，只有「{NAMES[b]}」对 {losses} 道，{p_value(sign_test(wins, losses))}"

print(f"留出集 {len(ROWS)} 道题，全部遮挡：")
print(f"{'':<18}{'第 1 名':<24}{'前 10 名'}")
for system in ("llm-direct", "phenotype-rank", "lirical", "rank-then-check", "fixed-pipeline", "agent"):
    print(f"{NAMES[system]:<16}{rate(system, 1):<24}{rate(system, 10)}")
print("\n主要结论（第 1 名）：", paired("agent", "lirical", 1))
print("前 10 名：          ", paired("agent", "lirical", 10))
print("\nLIRICAL 对我们自己写的两个不用模型的系统：")
for other in ("phenotype-rank", "rank-then-check"):
    for k in (1, 10):
        print(f"   {'第 1 名' if k == 1 else '前 10 名'}：", paired("lirical", other, k))

主要结论再从 MLflow 的考试记录里直接算一次（读两次考试的 840 条记录，约一分钟）：

In [ ]:
run("uv run raredx eval compare 5df45fbdae844d70a2631f751b8ec3b6 e1e02e75f67840c89bd88937aa2068a5 --k 1 2>/dev/null")

### 结果怎么看

| 留出集，840 道，全部遮挡 | 第 1 名 | 前 10 名 |
|---|---|---|
| 大模型直接猜 | 14.3% | 29.0% |
| 传统方法：按症状排名 | 17.5% | 44.4% |
| **LIRICAL** | **30.1%** [27.1–33.3] | **48.3%** [45.0–51.7] |
| 先排名再逐条核对 | 32.1% | 51.5% |
| 固定流程 | 36.8% | 51.5% |
| **最终 agent** | **42.4%** [39.1–45.8] | **62.0%** [58.7–65.2] |

**主要结论成立**：第 1 名，只有 agent 对的 121 道，只有 LIRICAL 对的 18 道，p < 0.001。前 10 名是 130 比 15。agent 比一个公认的工具，第 1 名多 12.3 个百分点。

两个 MLflow 记录算出来的也是 121 比 18：文件和记录对得上。

**LIRICAL 和我们自己写的对照比**：

- 比"传统方法"（按症状相似度排名）强得多：第 1 名 137 比 31。似然比这套方法，确实比单纯的相似度好
- 和"先排名再逐条核对"在第 1 名上没有显著差别（35 比 52，p = 0.086）；前 10 名上反而是我们的更好（26 比 53，p = 0.003）

后一条很重要：**我们的对照不是稻草人**。一个公认的工具，在同样条件下，也就是"先排名再逐条核对"的水平。agent 胜过这条对照，就等于胜过了这个水平的现成工具。

### 面试怎么说
> "Under the same masking, the final agent beats LIRICAL at first place by 12 points — 121 cases to 18, paired. LIRICAL beats plain semantic similarity by a lot, and is level with my strongest no-model baseline, which tells me that baseline was not a strawman."

---

## 第 7 步 · 不遮挡会怎样：泄露到底值多少分

### 在干什么
同一个工具、同一份题，遮一次、不遮一次，量出"答案就在知识库里"值多少分。

### 背景
这一步是这一章最有价值的发现，价值不在我们赢了多少，而在于它说明了一件更普遍的事：**很多罕见病诊断工具的成绩，是在这批公开病例上、不遮挡的条件下量出来的。** 如果泄露的影响很大，那些成绩就不能拿来和"遮挡后"的成绩比，也不能当成工具在新病人身上的真实水平。

第 10 章我们在开发集上第一次看到泄露。这里在 840 道留出集的题上，用三个系统（包括别人的工具）把它量清楚。

### 用了什么技术，为什么
一种很简单的实验设计：**只改一个条件，别的都不变**。同一个程序、同一份题、同一个判分，只有"遮不遮考题自己的论文"不同。两次的差，就是这个条件的作用。

再加一个**自然对照组**：840 道题里，有的题 HPO 根本没引用过病人自己的论文。对这些题，遮和不遮在知识库里没有任何区别，分数也应该一模一样。如果不一样，说明我们的遮挡做了不该做的事。

### 公司里什么情况怎么做
- **大模型评测里这个问题叫"数据污染"**（contamination）：考题出现在训练数据里，分数就虚高。各家公司发布模型时都要说明怎么查污染
- **训练集和测试集重叠的检查**是任何模型上线前的必查项
- **给分数注明条件**："在某某条件下测得"。条件不同的分数不能放在一张表里比

### 执行

In [ ]:
print(f"{'':<20}{'遮挡':<12}{'不遮挡':<12}{'差'}")
for system in ("phenotype-rank", "rank-then-check", "lirical"):
    masked = sum(r[f"{system}@1"] == "1" for r in ROWS) / len(ROWS)
    open_book = sum(r[f"{system} (unmasked)@1"] == "1" for r in ROWS) / len(ROWS)
    print(f"{NAMES[system]:<18}{masked:<12.1%}{open_book:<12.1%}+{(open_book - masked) * 100:.1f} 个百分点   （第 1 名）")
print("\n逐题比，LIRICAL：", paired("lirical (unmasked)", "lirical", 1))
print("都不遮，同等条件：", paired("rank-then-check (unmasked)", "lirical (unmasked)", 1))

cited = [r for r in ROWS if r["own_paper_cited"] == "1"]
not_cited = [r for r in ROWS if r["own_paper_cited"] == "0"]
print("\n按「HPO 有没有引用病人自己的论文」分开（LIRICAL，第 1 名）：")
print(f"   引用了     {len(cited)} 道：遮 {rate('lirical', 1, cited)}   不遮 {rate('lirical (unmasked)', 1, cited)}")
print(f"   没有引用   {len(not_cited)} 道：遮 {rate('lirical', 1, not_cited)}   不遮 {rate('lirical (unmasked)', 1, not_cited)}")
changed = sum(r["lirical@1"] != r["lirical (unmasked)@1"] for r in not_cited)
print(f"   没有引用的那些题里，遮和不遮结果不同的：{changed} 道")

alone = [r for r in ROWS if r["answerable"] == "0"]
print(f"\n病在 HPO 里只来自病人自己论文的 {len(alone)} 道题（遮挡后注定答不对）：")
for system in ("lirical (unmasked)", "rank-then-check (unmasked)", "lirical", "agent"):
    print(f"   {NAMES[system]:<22} 第 1 名答对 {sum(r[system + '@1'] == '1' for r in alone)} 道")

### 结果怎么看

**三个系统，不遮挡都高出 30 个百分点以上**（第 1 名）：

| | 遮挡 | 不遮挡 | 差 |
|---|---|---|---|
| 传统方法 | 17.5% | 48.0% | +30.5 |
| 先排名再逐条核对 | 32.1% | 67.9% | +35.7 |
| LIRICAL | 30.1% | 63.2% | +33.1 |

逐题看，LIRICAL 不遮挡多答对 278 道，**少答对 0 道**。不遮挡没有让任何一道题变难，只是让 278 道题变成了"在知识库里找到这个病人自己"。

**分数是从哪里多出来的**：

| | 题数 | LIRICAL 遮 | LIRICAL 不遮 |
|---|---|---|---|
| HPO 引用了病人自己的论文 | 630 | 32.2% | 76.3% |
| 没有引用 | 210 | 23.8% | 23.8% |

- 840 道题里，有 **630 道（75%）**，HPO 给这个病做标注时引用了病人自己的那篇论文
- 没有引用的 210 道，遮不遮**一道题都没变**。这是自然对照组：遮挡只影响它该影响的题
- 多出来的分数，全部来自那 630 道

**最极端的是那 122 道"病在 HPO 里只来自病人自己的论文"的题**：遮挡以后，LIRICAL 和 agent 都是 0 道；不遮挡，LIRICAL 第 1 名答对 83 道，先排名再逐条核对答对 100 道。这些"答对"，本质上是知识库里那个病的描述就是照着这个病人写的。

**同等条件下（都不遮）**，我们的"先排名再逐条核对"仍然比 LIRICAL 好：68 比 29。

**这意味着什么**：在这批公开病例上、不遮挡地评测一个诊断工具，第 1 名的成绩大约有一半是泄露。遮挡不是我们给自己加的额外难度，是让分数有意义的前提。

### 面试怎么说
> "The same tool on the same 840 cases scores 63% top-1 with each case's own paper in the knowledge base and 30% with it masked: 278 cases gained, none lost. Three quarters of these published cases are cited by HPO's annotation of their own disease, and the whole gain comes from those; on the other quarter, masking changes not a single answer — a built-in control. So a benchmark on public phenopackets that doesn't mask is roughly half leakage."

---

## 第 8 步 · 和考前的预期对照；下一步

### 在干什么
对答案；看看两个系统答对的题重不重合；说清楚下一步要比的 DeepRare 是什么情况。

### 执行

In [ ]:
both = lambda k: sum(r[f"agent@{k}"] == "1" or r[f"lirical@{k}"] == "1" for r in ROWS)
for k in (1, 10):
    agent = sum(r[f"agent@{k}"] == "1" for r in ROWS)
    lirical_hits = sum(r[f"lirical@{k}"] == "1" for r in ROWS)
    print(f"{'第 1 名' if k == 1 else '前 10 名'}：agent 答对 {agent} 道，LIRICAL {lirical_hits} 道，"
          f"两个里任一个答对 {both(k)} 道（比 agent 自己多 {both(k) - agent} 道）")

### 结果怎么看

**预期对照**：

| 考前的预期 | 实际 | 对吗 |
|---|---|---|
| LIRICAL 遮挡后第 1 名 30% 上下（28%–34%），前 10 名 50% 上下 | 30.1% 和 48.3% | ✅ |
| agent 第 1 名领先约 10 个百分点，前 10 名也是 10 个左右 | 12.3 个和 13.7 个，略多 | ✅ |
| LIRICAL 不遮挡到 65% 上下，比遮挡高 30 个百分点以上 | 63.2%，高 33.1 个 | ✅ |

这次三条都对了。原因不神秘：每一条都在开发集上先看过一次（第 5 步），留出集只是在更多的题上确认。第 21 章错的那两条，恰恰是开发集上看不准的东西（领先会不会缩水）。

**两个系统答对的题，大部分重合**：第 1 名，两个里任一个答对的只比 agent 自己多 18 道。所以"把 LIRICAL 加给 agent 当工具"能多答对的题不多。这次只看，不改系统（方案里写好的）。

**下一步：DeepRare**

DeepRare 是 2025 年发表的罕见病诊断 agent，和我们的项目最像。我读了它的代码（`third_party/DeepRare`），要比它，有三件事要先说清楚：

| | 情况 |
|---|---|
| **要花钱** | 每道题要调用大模型 25 到 60 次，很多次带着整篇网页或摘要。默认用的是推理模型（DeepSeek-R1），推理模型的输出很长，单价还要高。另外它要一个 OpenAI 的密钥算文本向量 |
| **很慢** | 每道题要上网搜索、用浏览器打开网页、查 PubMed，估计要几分钟。840 道题要跑几十个小时 |
| **会泄露** | 它的工具会上网搜索、查 PubCaseFinder 和 PhenoBrain、查一个 6.7 万例的 PubMed 病例库。这些地方都可能直接读到考题病人自己的论文。第 7 步刚看到，这种泄露能值 30 多个百分点 |

所以 DeepRare 不能照搬 LIRICAL 的做法。我的建议是：先拿 3 道题试跑，量出每道题的花费和时间；再从 840 道里随机抽 100 道来比，并且记下它每道题读到了哪些资料、有没有读到病人自己的论文。花费和范围要你来定。

### 面试怎么说
> "All three predictions held this time, because each had been checked on the dev set first. The next comparison, DeepRare, is a different problem: it searches the web, PubCaseFinder and a 67k-case PubMed bank, so it can read the patient's own report. Comparing it fairly means logging what it retrieved for each case and counting how often that included the source paper — not just copying its published numbers."

---

## 本章小结

| 步骤 | 做了什么 | 结果 |
|---|---|---|
| 选对手 | 三个条件：够强、做同一件事、能在同样条件下考 | LIRICAL 2.4.1；别的工具为什么不比，写明 |
| 接进考场 | 适配器；版本和校验码登记 | 和我们的系统走同一套考试、判分、泄露审计 |
| 同样的遮挡 | 每道题单独写一份遮过的知识库文件 | 改输入，不改它的程序 |
| 先写方案 | `docs/comparison_plan.md` | 一个主要结论；4 次考试；考前的预期 |
| 开发集上调通 | 修好启动崩溃；CPU 减半 | 输出都核对过，不变 |
| **主要结论** | 同样遮挡，第 1 名 | **agent 42.4%，LIRICAL 30.1%；121 比 18，p < 0.001** |
| 对照是不是稻草人 | LIRICAL 对我们自己写的对照 | 和"先排名再逐条核对"打平 |
| **泄露值多少分** | 同一个工具遮和不遮 | **LIRICAL 63.2% 对 30.1%**；多出来的全部来自 HPO 引用了病人自己论文的那 75% 的题 |
| 花费 | | 0 元 |

## 一句话

**和别人比，要把别人请进自己的考场**：同样的题、同样的知识库、同样的遮挡。照抄别人论文里的分数，比的是两种不同的考试。

## 面试题

1. **How does your agent compare with existing tools?**
   > "I ran LIRICAL, a published and maintained tool, on the same 840 held-out cases under the same leakage controls: 30% top-1 against the agent's 42%, 121 cases to 18 paired. LIRICAL is level with my strongest no-model baseline, so the baselines in my main evaluation were not strawmen."

2. **Why not just cite LIRICAL's published accuracy?**
   > "Because it was measured differently. With each case's own paper left in the knowledge base, LIRICAL scores 63% on the same cases. Numbers measured with and without that leak cannot go in one table."

3. **How do you mask a tool you can't modify?**
   > "Through its input: for each case it reads its own copy of the annotation file, written under the same three rules my tools apply. Its output is traced, so the same leak audit reads it. On the quarter of cases whose paper HPO doesn't cite, masking changed no answer at all — a built-in check that it removes only what it should."

4. **What did leakage do to the numbers?**
   > "About half of an unmasked top-1 score on these public cases is leakage: plus 30 to 36 points for all three systems I measured, coming entirely from the 75% of cases whose source paper the annotations cite."

**下一步**（还没开始，等你决定）：DeepRare 先试跑 3 道题量花费；通用大模型通过 OpenRouter 一处充值、在 20 道题上先量单价。两件事都要先定预算。